In [2]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA

In [3]:
df=pd.read_csv('uber.csv')

df.head()

,Index,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              200000 non-null  int64  
 1   key                200000 non-null  object 
 2   fare_amount        200000 non-null  float64
 3   pickup_datetime    200000 non-null  object 
 4   pickup_longitude   200000 non-null  float64
 5   pickup_latitude    200000 non-null  float64
 6   dropoff_longitude  199999 non-null  float64
 7   dropoff_latitude   199999 non-null  float64
 8   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 13.7+ MB


In [5]:
print(df.shape)
df.describe()

(200000, 9)


,Index,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+05,200000.000000,200000.000000,200000.000000,199999.000000,199999.000000,200000.000000
mean,2.771250e+07,11.359955,-72.527638,39.935885,-72.525292,39.923890,1.684535
std,1.601382e+07,9.901776,11.437787,7.720539,13.117408,6.794829,1.385997
min,1.000000e+00,-52.000000,-1340.648410,-74.015515,-3356.666300,-881.985513,0.000000
25%,1.382535e+07,6.000000,-73.992065,40.734796,-73.991407,40.733823,1.000000
50%,2.774550e+07,8.500000,-73.981823,40.752592,-73.980093,40.753042,1.000000
75%,4.155530e+07,12.500000,-73.967154,40.767158,-73.963658,40.768001,2.000000
max,5.542357e+07,499.000000,57.418457,1644.421482,1153.572603,872.697628,208.000000


In [6]:
df=df[df['fare_amount']>0.0]
df.shape

(199978, 9)

In [7]:
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]
df = df[(df['pickup_latitude'] >= -90) & (df['pickup_latitude'] <= 90)]
df = df[(df['dropoff_latitude'] >= -90) & (df['dropoff_latitude'] <= 90)]
df = df[(df['pickup_longitude'] >= -180) & (df['pickup_longitude'] <= 180)]
df = df[(df['dropoff_longitude'] >= -180) & (df['dropoff_longitude'] <= 180)]
df.shape

(199256, 9)

In [8]:
df.isnull().sum()

Index                0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

In [9]:
df['pickup_datetime']=pd.to_datetime(df['pickup_datetime'],errors='coerce')
df.dtypes

Index                              int64
key                               object
fare_amount                      float64
pickup_datetime      datetime64[ns, UTC]
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dtype: object

In [10]:
df['hour']=df['pickup_datetime'].dt.hour
df['weekday']=df['pickup_datetime'].dt.weekday
df['month']=df['pickup_datetime'].dt.month
df['year']=df['pickup_datetime'].dt.year

df['is_weekday']=df['weekday'].isin([5,6]).astype(int)
df['is_night']=df['hour'].isin([22,23,0,1,2,3,4,5]).astype(int)



In [11]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['distance_km'] = haversine(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)
df = df.drop(columns=['key', 'pickup_datetime'])

df.head()

,Index,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,weekday,month,year,is_weekday,is_night,distance_km
0,24238194,7.5,-73.999817,40.738354,-73.999512,40.723217,1,19,3,5,2015,0,0,1.683323
1,27835199,7.7,-73.994355,40.728225,-73.994710,40.750325,1,20,4,7,2009,0,0,2.457590
2,44984355,12.9,-74.005043,40.740770,-73.962565,40.772647,1,21,0,8,2009,0,0,5.036377
3,25894730,5.3,-73.976124,40.790844,-73.965316,40.803349,3,8,4,6,2009,0,0,1.661683
4,17610152,16.0,-73.925023,40.744085,-73.973082,40.761247,5,17,3,8,2014,0,0,4.475450


In [12]:
# codf=df[['fare_amount','distance_km','is_night','is_weekday','month','year']].corr()
df.head()
df.to_csv('cleaned_uber.csv')
# sns.heatmap(codf,annot=True)
# # plt.plot()
# df=df.map({'weekday':'day'})

In [13]:
x=df[['passenger_count','hour','weekday','month','year','is_weekday','is_night','distance_km']]
y=df['fare_amount']


x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

preprocess=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['passenger_count','hour','weekday','month','year','distance_km']),
        ('cat',OneHotEncoder(),['is_night','is_weekday'])
    ]
)
pipe=Pipeline([
    ('step',preprocess),
    ('model',RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=3,
    max_features="sqrt"
))
])
pipe.fit(x_train,y_train)
pred=pipe.predict(x_test)

print('Accuracy : ',np.round(r2_score(y_test,pred)*100,2),'%')
print('MSE : ',np.round(mean_squared_error(y_test,pred),2))



Accuracy :  75.8 %
MSE :  23.48


In [14]:
import joblib
joblib.dump(pipe,'UberFarePredictor.pkl',compress=3)

['UberFarePredictor.pkl']

In [15]:
import joblib
import pandas as pd
model=joblib.load('UberFarePredictor.pkl')
data = {
    'passenger_count': [2],
    'hour': [12],
    'weekday': [1],
    'month': [12],
    'year': [2015], 
    'is_weekday':[0] ,
    'is_night': [0],
    'distance_km': [5]
}

df = pd.DataFrame(data)

result = model.predict(df)
print("Predicted Fare:", np.round(result[0],2))




Predicted Fare: 18.93
